<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula06a_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"accuracy: {accuracy_score(y_test, y_pred)}")

accuracy: 0.9444444444444444


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, cross_validate, KFold

spliter = KFold(n_splits=5, shuffle=True)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

param_grid = {
    'scaler__with_mean': [True, False],
    'scaler__with_std': [True, False],
    'model__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
    'model__metric': ['euclidean', 'manhattan', 'minkowski', 'chebyshev'],
    'model__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'model__weights': ['uniform', 'distance']
    }

grid_search = RandomizedSearchCV(pipe, param_grid, cv=spliter, verbose=1, n_iter=200)
scores = cross_validate(grid_search, X, y, cv=spliter, return_train_score=True)
print(f"train accuracy: {scores['train_score'].mean()}")
print(f"test accuracy: {scores['test_score'].mean()}")


Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Fitting 5 folds for each of 200 candidates, totalling 1000 fits
train accuracy: 0.9971830985915492
test accuracy: 0.9666666666666668


In [3]:
!pip install optuna -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 18.1 MB/s eta 0:00:00


In [4]:
import optuna

def objective(trial):
  # Define hyperparameters for StandardScaler
  with_mean = trial.suggest_categorical('with_mean', [True, False])
  with_std = trial.suggest_categorical('with_std', [True, False])
  scaler = StandardScaler(with_mean=with_mean, with_std=with_std)

  # Define hyperparameters for KNeighborsClassifier
  n_neighbors = trial.suggest_int('n_neighbors', 1, 15)
  metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski', 'chebyshev'])
  algorithm = trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
  weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
  model = KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric, algorithm=algorithm, weights=weights)

  pipeline = Pipeline([
      ('scaler', scaler),
      ('model', model)
  ])

  spliter = KFold(n_splits=5, shuffle=True)
  scores = cross_validate(pipeline, X_train, y_train, cv=spliter)
  return scores['test_score'].mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

print(f"Best accuracy: {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")
#

[I 2026-09-01 20:04:10,242] A new study created in memory with name: no-name-bf028bd1-e937-452f-b4b7-3ae9d4802cc9
[I 2026-09-01 20:04:10,269] Trial 0 finished with value: 0.6916256157635468 and parameters: {'with_mean': False, 'with_std': False, 'n_neighbors': 4, 'metric': 'euclidean', 'algorithm': 'brute', 'weights': 'distance'}. Best is trial 0 with value: 0.6916256157635468.
[I 2026-09-01 20:04:10,299] Trial 1 finished with value: 0.9226600985221675 and parameters: {'with_mean': False, 'with_std': True, 'n_neighbors': 4, 'metric': 'chebyshev', 'algorithm': 'kd_tree', 'weights': 'distance'}. Best is trial 1 with value: 0.9226600985221675.
[I 2026-09-01 20:04:10,328] Trial 2 finished with value: 0.7391625615763546 and parameters: {'with_mean': False, 'with_std': False, 'n_neighbors': 7, 'metric': 'chebyshev', 'algorithm': 'auto', 'weights': 'distance'}. Best is trial 1 with value: 0.9226600985221675.
[I 2026-09-01 20:04:10,356] Trial 3 finished with value: 0.9719211822660098 and param

Best accuracy: 0.993103448275862
Best hyperparameters: {'with_mean': False, 'with_std': True, 'n_neighbors': 1, 'metric': 'manhattan', 'algorithm': 'brute', 'weights': 'distance'}


In [9]:
from sklearn.base import BaseEstimator, TransformerMixin

class AutoKNN(BaseEstimator, TransformerMixin):
  def __init__(self, n_trials=200):
    self.n_trials = n_trials
    self.study = None
  def fit(self, X, y=None):
    def objective(trial):
      # Define hyperparameters for StandardScaler
      with_mean = trial.suggest_categorical('with_mean', [True, False])
      with_std = trial.suggest_categorical('with_std', [True, False])
      # Define hyperparameters for KNeighborsClassifier
      n_neighbors = trial.suggest_int('n_neighbors', 1, 15)
      metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski', 'chebyshev'])
      algorithm = trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
      weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
      pipeline = Pipeline([
          ('scaler', StandardScaler(with_mean=with_mean, with_std=with_std)),
          ('model', KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric, algorithm=algorithm, weights=weights))
      ])
      spliter = KFold(n_splits=5, shuffle=True)
      scores = cross_validate(pipeline, X_train, y_train, cv=spliter)
      return scores['test_score'].mean()
    self.study = optuna.create_study(direction='maximize')
    self.study.optimize(objective, n_trials=self.n_trials)
    best_params = self.study.best_params
    scaler = StandardScaler(with_mean=best_params['with_mean'], with_std=best_params['with_std'])
    model = KNeighborsClassifier(n_neighbors=best_params['n_neighbors'], metric=best_params['metric'], algorithm=best_params['algorithm'], weights=best_params['weights'])
    self.pipeline = Pipeline([
        ('scaler', scaler),
        ('model', model)
    ])
    self.pipeline.fit(X, y)
    return self
  def predict(self, X):
    return self.pipeline.predict(X)
  def score(self, X, y):
    return self.pipeline.score(X, y)



In [10]:
scores = cross_validate(AutoKNN(), X, y, cv=spliter, return_train_score=True)
print(f"train accuracy: {scores['train_score'].mean()}")
print(f"test accuracy: {scores['test_score'].mean()}")

[I 2026-09-01 20:05:56,693] A new study created in memory with name: no-name-17de9e05-6f85-42a9-b517-98ae27d31b7f
[I 2026-09-01 20:05:56,739] Trial 0 finished with value: 0.9433497536945813 and parameters: {'with_mean': False, 'with_std': True, 'n_neighbors': 3, 'metric': 'minkowski', 'algorithm': 'brute', 'weights': 'distance'}. Best is trial 0 with value: 0.9433497536945813.
[I 2026-09-01 20:05:56,831] Trial 1 finished with value: 0.9573891625615764 and parameters: {'with_mean': True, 'with_std': True, 'n_neighbors': 3, 'metric': 'euclidean', 'algorithm': 'kd_tree', 'weights': 'distance'}. Best is trial 1 with value: 0.9573891625615764.
[I 2026-09-01 20:05:56,885] Trial 2 finished with value: 0.9226600985221675 and parameters: {'with_mean': True, 'with_std': True, 'n_neighbors': 11, 'metric': 'chebyshev', 'algorithm': 'brute', 'weights': 'distance'}. Best is trial 1 with value: 0.9573891625615764.
[I 2026-09-01 20:05:56,930] Trial 3 finished with value: 0.9647783251231526 and paramet

train accuracy: 1.0
test accuracy: 0.9776190476190475
